# 02 - Baseline Experiments

This notebook trains and evaluates two baseline methods:
1. Logistic Regression on flattened pixel features
2. Shallow CNN trained from scratch

Both baselines serve as reference points for the more advanced models.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from src.data import get_flat_features, get_dataloaders
from src.models import get_model, get_optimizer, FocalLoss
from src.train import run_experiment
from src.evaluate import (
    compute_metrics, evaluate_model, find_optimal_threshold,
    plot_confusion_matrix, plot_roc_curve, plot_training_history,
)
from src.utils import set_seed, get_device, load_config, save_results

plt.rcParams["figure.dpi"] = 100
sns.set_style("whitegrid")

config = load_config("../configs/default.yaml")
device = get_device()
DATA_ROOT = "../data/chest_xray"
SEED = config["reproducibility"]["seed"]

print(f"Device: {device}")

## Baseline 1: Logistic Regression

We flatten 64x64 grayscale images into 4096-dimensional vectors
and train an L2-regularized logistic regression classifier.

In [ ]:
set_seed(SEED)

print("Loading flattened features...")
flat_data = get_flat_features(DATA_ROOT, image_size=64, seed=SEED)

print(f"Train: {flat_data['X_train'].shape}")
print(f"Val:   {flat_data['X_val'].shape}")
print(f"Test:  {flat_data['X_test'].shape}")

In [ ]:
print("Training Logistic Regression...")
lr_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver="lbfgs",
    class_weight="balanced",
    random_state=SEED,
)
lr_model.fit(flat_data["X_train"], flat_data["y_train"])

val_proba_lr = lr_model.predict_proba(flat_data["X_val"])[:, 1]
test_proba_lr = lr_model.predict_proba(flat_data["X_test"])[:, 1]

val_metrics_lr = compute_metrics(flat_data["y_val"], val_proba_lr)
test_metrics_lr = compute_metrics(flat_data["y_test"], test_proba_lr)

print(f"\nValidation - AUROC: {val_metrics_lr['auroc']:.4f}, F1: {val_metrics_lr['f1_macro']:.4f}")
print(f"Test       - AUROC: {test_metrics_lr['auroc']:.4f}, F1: {test_metrics_lr['f1_macro']:.4f}")
print(f"             Sensitivity: {test_metrics_lr['sensitivity']:.4f}, "
      f"Specificity: {test_metrics_lr['specificity']:.4f}")

In [ ]:
lr_results = {
    "metrics": test_metrics_lr,
    "y_true": flat_data["y_test"],
    "y_proba": test_proba_lr,
}

fig, ax = plt.subplots(figsize=(5, 4))
plot_confusion_matrix(
    flat_data["y_test"], test_proba_lr,
    title="Logistic Regression - Test Set", ax=ax,
)
plt.tight_layout()
plt.savefig("../results/baseline_lr_confusion.png", bbox_inches="tight")
plt.show()

## Baseline 2: Shallow CNN

A 3-layer CNN trained from scratch with weighted binary cross-entropy.

In [ ]:
set_seed(SEED)

dataloaders = get_dataloaders(
    DATA_ROOT,
    augmentation="basic",
    image_size=config["data"]["image_size"],
    batch_size=config["training"]["batch_size"],
    val_split=config["data"]["val_split"],
    num_workers=0,
    seed=SEED,
)

print(f"Train batches: {len(dataloaders['train'])}")
print(f"Val batches:   {len(dataloaders['val'])}")
print(f"Test batches:  {len(dataloaders['test'])}")

In [ ]:
set_seed(SEED)

cnn_model = get_model("shallow_cnn", dropout=config["model"]["dropout"])

cnn_config = config.copy()
cnn_config["model"]["use_focal_loss"] = False

cnn_results = run_experiment(
    model_name="shallow_cnn",
    model=cnn_model,
    dataloaders=dataloaders,
    device=device,
    config=cnn_config,
    experiment_name="baseline_shallow_cnn",
)

In [ ]:
fig = plot_training_history(cnn_results["history"], title="Shallow CNN")
plt.savefig("../results/baseline_cnn_training.png", bbox_inches="tight")
plt.show()

In [ ]:
cnn_test = evaluate_model(cnn_model, dataloaders["test"], device)

print("Shallow CNN - Test Results:")
print(f"  AUROC:       {cnn_test['metrics']['auroc']:.4f}")
print(f"  F1 (macro):  {cnn_test['metrics']['f1_macro']:.4f}")
print(f"  Sensitivity: {cnn_test['metrics']['sensitivity']:.4f}")
print(f"  Specificity: {cnn_test['metrics']['specificity']:.4f}")
print(f"  NPV:         {cnn_test['metrics']['npv']:.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
plot_confusion_matrix(
    cnn_test["y_true"], cnn_test["y_proba"],
    title="Shallow CNN - Test Set", ax=ax,
)
plt.tight_layout()
plt.savefig("../results/baseline_cnn_confusion.png", bbox_inches="tight")
plt.show()

## Summary of Baseline Results

In [ ]:
import pandas as pd

baseline_results = {
    "Logistic Regression": lr_results,
    "Shallow CNN": cnn_test,
}

summary_rows = []
for name, res in baseline_results.items():
    m = res["metrics"]
    summary_rows.append({
        "Model": name,
        "AUROC": f"{m['auroc']:.4f}",
        "F1 (macro)": f"{m['f1_macro']:.4f}",
        "Sensitivity": f"{m['sensitivity']:.4f}",
        "Specificity": f"{m['specificity']:.4f}",
        "NPV": f"{m['npv']:.4f}",
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

In [ ]:
save_results(
    {"logistic_regression": test_metrics_lr, "shallow_cnn": cnn_test["metrics"]},
    "baseline_results",
    output_dir="../results",
)
print("Baseline results saved.")